# 04 · Metingen publiceren met MQTT

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Tot nu toe staat jouw database (`asserva.db` of `schauer.db`) alleen in Codespaces. Om die gegevens door te geven aan een andere computer, moet Python de rijen eerst uitlezen en als berichten versturen via jouw Mosquitto-broker op DigitalOcean.

Je begint met één klein zelfgemaakt oefenbericht. Daarna verstuur je twintig echte meetrijen uit jouw database en onderzoek je wat er gebeurt als je dezelfde selectie nog eens aanbiedt, zodat je het hele mechanisme goed begrijpt voordat je in het volgende notebook de volledige database (`asserva.db` of `schauer.db`) deelt.

**Nodig:** je eigen branch en Codespace uit notebook 01, je aangevulde werkdatabase uit notebook 02, je ingerichte server uit notebook 03 (met `gegevens/werk/server.json` en `gegevens/werk/ca.crt`), en jouw **`MQTT-broker eigenaar`**-wachtwoord (uit notebook 03).

Voer de codecellen één voor één van boven naar beneden uit en beantwoord tussendoor de vragen. *(Let op: wanneer een cel om jouw wachtwoord vraagt, verschijnt het invoerbalkje helemaal bovenaan in het venster van Codespaces/Cursor.)*

</div>

## 1. Je werkplek en server hervatten

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 04)

De **rol** die je in notebook 01 koos, bepaalt opnieuw welke database je gebruikt: **`asserva.db`** (meetkolom `VBW` in kg) bij gewichten, of **`schauer.db`** (meetkolom `VFI` in g) bij voederopname.

De cel hieronder controleert je opgeslagen instellingen en kijkt of jouw werkdatabase, `server.json` en `ca.crt` klaarstaan. Ontbreekt de werkdatabase nog, rond dan eerst notebook 02 af.

</div>

In [ ]:
from pathlib import Path
from contextlib import closing
from getpass import getpass
from datetime import datetime, timezone
import subprocess
import sys
import json
import re
import ipaddress

root = subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True)
if root.returncode:
    raise RuntimeError("Open notebook 04 vanuit de practicumrepository in Codespaces.")
hoofdmap = Path(root.stdout.strip())
branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=hoofdmap, text=True).strip()
instellingen = json.loads((hoofdmap / "gegevens/inlevering/01_werkplekcontrole.json").read_text(encoding="utf-8"))
deelnemer = instellingen["deelnemerscode"]
if not re.fullmatch(r"[a-z0-9][a-z0-9_-]{0,62}", deelnemer) or branch != "student/" + deelnemer or instellingen.get("branch") != branch:
    raise ValueError("Controleer je eigen branch en werkplekcontrole uit notebook 01.")
rol = instellingen["rol"]
bestanden = {"gewichten": "asserva.db", "voederopname": "schauer.db"}
if rol not in bestanden or instellingen.get("bronbestand") != bestanden[rol]:
    raise ValueError("Rol en databank komen niet overeen. Controleer notebook 01.")
werkmap = hoofdmap / "gegevens/werk"
werkdatabase = werkmap / bestanden[rol]
server = json.loads((werkmap / "server.json").read_text(encoding="utf-8"))
if server.get("deelnemerscode") != deelnemer:
    raise ValueError("Dit serverbestand hoort bij een andere deelnemer.")
serveradres = str(ipaddress.IPv4Address(server["ipv4"]))
ca = werkmap / "ca.crt"
for pad in (werkdatabase, ca):
    if not pad.is_file():
        raise FileNotFoundError(f"Ontbreekt: {pad.relative_to(hoofdmap)}. Hervat notebook 02 of 03.")
sys.path.insert(0, str(hoofdmap / "programmatuur"))
from publiceren import MqttSessie, lees_metingen, meettopic, publiceer_selectie
print("Deelnemer:", deelnemer, "| Rol:", rol)
print("Werkdatabase:", werkdatabase.relative_to(hoofdmap))
print("SSH-opdracht:", f"ssh -o HostKeyAlgorithms=ssh-ed25519 beheer@{serveradres}")

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Terminal), vervolgens op je DigitalOcean-server (SSH-terminal, als beheer)

1. Open in Codespaces een terminal via **Terminal → New Terminal** en voer de `ssh`-opdracht uit die hierboven is afgedrukt *(werk je lokaal op Windows of macOS, gebruik dan PowerShell, Git Bash of Terminal)*. Log in met jouw **`SSH beheer-wachtwoord`** uit notebook 03.
2. Start Mosquitto (als die na een serverherstart nog uitstond) en bekijk zijn status:

</div>

```bash
sudo systemctl start mosquitto
sudo systemctl status mosquitto --no-pager
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

3. Controleer dat je een groen bolletje (`●`) en **`Active: active (running)`** ziet. Daarna mag je deze SSH-terminal open laten of weer sluiten, want Mosquitto blijft gewoon op je Droplet draaien.

</div>

<div style="background-color:#fff8e1;color:#6d4c00;border-left:5px solid #f9a825;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Let op: welk wachtwoord en waar vul je het in?</strong><br>Het <strong><code>SSH beheer-wachtwoord</code></strong> gebruik je alleen in de terminal voor serverbeheer. In de Python-cellen van dit notebook gebruik je jouw <strong><code>MQTT-broker eigenaar</code></strong>-wachtwoord (uit notebook 03). Wanneer een cel daarom vraagt, verschijnt een klein invoerbalkje <strong>helemaal bovenaan in het venster</strong> van Codespaces/Cursor (niet onder de cel zelf, en zonder sterretjes). Typ of plak het wachtwoord daar en druk op Enter, anders blijft de cel wachten. Sla wachtwoorden nooit op in een cel, antwoord of Git-commit.</div>

## 2. Wat wordt er uitgewisseld?

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 04)

Een **publisher** verstuurt een bericht en een **subscriber** vraagt berichten op een bepaald topic aan. Beide zijn MQTT-clients die verbinden met een **broker**. De broker ontvangt een bericht en geeft het door aan aangesloten clients met een passend abonnement (één client kan dus ook tegelijk versturen en ontvangen).

Een MQTT-bericht bestaat uit een **topic** en een **payload**:
- Het **topic** is het adres van de berichtenstroom, bijvoorbeeld `practicum/voornaam_achternaam/metingen/gewichten`.
- De **payload** is de eigenlijke inhoud van het bericht (wij kiezen hier **JSON** als leesbaar formaat).

Belangrijk: een topic is géén SQL-tabel, maar alleen een adres waarop de broker berichten sorteert en doorstuurt. Mosquitto bewaart dus nog niets in een database. In dit notebook kijkt een tijdelijk abonnement in Python alleen mee om de ontvangst te controleren. In notebook 05 bouwen we daarna de echte ontvanger die berichten in SQLite opslaat.

</div>

| Onderdeel | Plaats | Functie in deze oefening |
|---|---|---|
| Werkdatabase | Codespace | Bewaart de meetrecords en hokmetadata |
| Publicerende Python-code | Codespace | Leest records en maakt berichten |
| Mosquitto | Eigen Droplet | Verdeelt berichten op basis van topics |
| Tijdelijk abonnement | Codespace | Laat ontvangen inhoud zien |
| Permanente ontvanger | Volgende oefening | Slaat ontvangen metingen in SQLite op |

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 1.** Waarom is alleen Mosquitto starten onvoldoende om je database te delen?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 1):**

Voeg hier je antwoord in. Benoem de taak van de publicerende code en die van de broker.

</div>

## 3. Zelf een JSON-bericht samenstellen

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 04)

**JSON** beschrijft gegevens met sleutel-waardeparen tussen accolades (`{ ... }`). Tekst staat tussen dubbele aanhalingstekens, terwijl getallen zonder aanhalingstekens staan en een punt als decimaalteken gebruiken.

In Python begin je met een `dict` (een verzameling sleutel-waardeparen in het geheugen van je programma). Met `json.dumps(...)` zet je die om naar **JSON-tekst**, zodat het bericht via het netwerk verstuurd kan worden. Let erop dat een ontbrekende waarde (`None` in Python) in JSON `null` wordt, wat iets anders is dan het getal `0`.

Hieronder maak je een klein oefenbericht over de oppervlakte van hok 1. Het tijdstip geeft aan wanneer je dit oefenbericht maakt (met een expliciete eenheid erbij zodat getallen altijd duidelijk blijven).

**Communicatie:** er vindt hier nog **geen netwerkverkeer** plaats. Python maakt dit oefenbericht alleen lokaal in het geheugen van jouw **Codespace** (en raakt jouw SQLite-database niet aan).

</div>

In [ ]:
oefenbericht = {
    "soort": "oefening",
    "hok": "1",
    "eigenschap": "oppervlakte",
    "waarde": 8.0,
    "eenheid": "m2",
    "gemaakt_op": datetime.now(timezone.utc).isoformat(),
}
json_voorbeeld = json.dumps(oefenbericht, ensure_ascii=False, indent=2, allow_nan=False)
print(json_voorbeeld)
print("Python-type:", type(oefenbericht).__name__)
print("Type na omzetting:", type(json_voorbeeld).__name__)

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 2.** Welke onderdelen van dit bericht zijn tekst, welke zijn getallen en waarom staat het hoknummer tussen aanhalingstekens?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 2):**

Voeg hier je antwoord in. Gebruik minstens één voorbeeld van elk type uit jouw uitvoer.

</div>

## 4. Publiceren en de ontvangst bekijken

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 04)

Je verstuurt dit oefenbericht naar een apart testtopic onder `controle/`, zodat het niet tussen je echte metingen belandt. De cel hieronder maakt met `MqttSessie` (aangeleverde code uit `programmatuur/publiceren.py`) een versleutelde verbinding via `ca.crt`, wacht tot de broker jouw abonnement bevestigt, stuurt het bericht uit en leest het weer terug. Na afloop sluit het `with`-blok de verbinding weer netjes af.

**Communicatie (`Codespaces → DigitalOcean → Codespaces`):** jouw Python-code in **Codespaces** abonneert zich eerst bij jouw **Mosquitto-broker op DigitalOcean**, stuurt het oefenbericht naar de broker (poort 8883) en ontvangt datzelfde bericht via het abonnement weer terug in **Codespaces**.

**Wachtwoord invoeren:** wanneer je deze cel uitvoert, verschijnt **helemaal bovenaan in het venster** van Codespaces/Cursor (niet onder de cel) een klein invoerbalkje. Vul daar jouw **`MQTT-broker eigenaar`**-wachtwoord in en druk op **Enter**. Dat wachtwoord wordt **niet opgeslagen**, maar alleen even in het werkgeheugen gebruikt om bij je broker aan te melden.

</div>

In [ ]:
oefentopic = f"practicum/{deelnemer}/controle/json"
with MqttSessie(serveradres, ca, getpass("Wachtwoord 'MQTT-broker eigenaar': "), abonnement=oefentopic) as verbinding:
    verbinding.verstuur(oefentopic, oefenbericht)
    ontvangen_oefening = verbinding.lees()
assert ontvangen_oefening["payload"] == oefenbericht
print("Ontvangen topic:", ontvangen_oefening["topic"])
print(json.dumps(ontvangen_oefening["payload"], ensure_ascii=False, indent=2))

<div style="background-color:#e8f5e9;color:#1b5e20;border-left:5px solid #2e7d32;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Controle</strong><br>Je ziet hetzelfde topic en dezelfde inhoud terug. Dit is daadwerkelijk ontvangen MQTT-verkeer. De vergelijking controleert de ontvangen inhoud, niet alleen of de publicatieopdracht zonder fout eindigde.</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Doe-opdracht:** maak in de cel hieronder een kopie van `oefenbericht` (`aangepast = dict(oefenbericht)`) en verander daarin alleen `waarde` naar `9.0`. Je past hiermee alleen dit tijdelijke Python-object aan, de echte hokoppervlakte (`8 m²`) in jouw database blijft ongewijzigd. Voer daarna ook de ontvangstcel eronder uit: er verschijnt opnieuw **bovenaan in het venster** een invoerbalkje voor jouw **`MQTT-broker eigenaar`**-wachtwoord (dat niet wordt opgeslagen, maar alleen dient om aan te melden).

</div>

In [ ]:
# Maak een kopie: aangepast = dict(oefenbericht)
# Vul daarna in: aangepast["waarde"] = ...

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Voer nu de cel hieronder uit om het aangepaste bericht te versturen en terug te lezen. **Communicatie (`Codespaces → DigitalOcean → Codespaces`):** het aangepaste Python-object gaat vanuit **Codespaces** naar de **Mosquitto-broker op DigitalOcean** en komt via het abonnement terug naar **Codespaces** (jouw SQLite-database wordt hierbij niet aangesproken). Vul in het invoerbalkje **bovenaan in het venster** opnieuw jouw **`MQTT-broker eigenaar`**-wachtwoord in en druk op **Enter** (het wachtwoord wordt niet opgeslagen, alleen gebruikt om verbinding te maken):

</div>

In [ ]:
oefentopic = f"practicum/{deelnemer}/controle/json"
with MqttSessie(serveradres, ca, getpass("Wachtwoord 'MQTT-broker eigenaar': "), abonnement=oefentopic) as verbinding:
    verbinding.verstuur(oefentopic, aangepast)
    ontvangen_aanpassing = verbinding.lees()
assert ontvangen_aanpassing["payload"] == aangepast
print("Ontvangen oefenwaarde:", ontvangen_aanpassing["payload"]["waarde"])
print("Eerste oefenwaarde:", oefenbericht["waarde"])

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 3.** Welke waarde ontving je na de aanpassing? Is daardoor de oppervlakte in de tabel `hokken` van jouw database gewijzigd?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 3):**

*Voeg hier je antwoord in. Tip: noteer (1) welke waarde je via MQTT hebt ontvangen, (2) of het getal in jouw SQLite-tabel `hokken` daardoor veranderd is, en (3) waarom wel of niet (heb je alleen een los bericht in Python verstuurd, of ook de database zelf bijgewerkt?).*

</div>

## 5. Een echt meetrecord lezen

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 04)

We beginnen niet meteen met alle tienduizenden metingen tegelijk, maar lezen eerst de eerste twintig rijen uit jouw werkdatabase van notebook 02 (`ORDER BY DATE, ROW_ID`).

De aangeleverde functie `lees_metingen` opent je database alleen om te lezen en maakt van elke rij een JSON-bericht met herkenbare Nederlandse velden. Het volledige oorspronkelijke record blijft daarnaast bewaard onder `bronrecord` (inclusief eventuele kwaliteitsvlaggen), en de hokmetadata uit jouw tabel `hokken` komen onder `hokmetadata`.

De combinatie van **`bron`** en **`record_id`** (met alle cijfers van `ROW_ID` als tekst) vormt de unieke identificatie van elke meting, zodat een ontvanger herhalingen later kan herkennen.

**Communicatie (`Codespaces-database → Python in Codespaces`):** Python leest deze twintig rijen alleen lokaal uit jouw SQLite-werkdatabase in **Codespaces** (`gegevens/werk/`). Er gaat hier nog **niets** naar de Mosquitto-broker op DigitalOcean.

</div>

In [ ]:
aantal_in_selectie = 20
selectie = list(lees_metingen(werkdatabase, rol, deelnemer, limiet=aantal_in_selectie))
if len(selectie) != aantal_in_selectie:
    raise ValueError("De werkdatabase bevat minder records dan de gevraagde selectie.")
topic_metingen = meettopic(deelnemer, rol)
eerste = selectie[0]
print("Topic:", topic_metingen)
print("Geselecteerde records:", len(selectie))
print(json.dumps(eerste, ensure_ascii=False, indent=2, allow_nan=False))

| Veld | Betekenis |
|---|---|
| `schema_versie` | Versie van de afgesproken berichtstructuur |
| `bron`, `record_id` | Stabiele combinatie die de bronrij identificeert |
| `deelnemer` | Deelnemerscode van de publicerende client |
| `meettype`, `meetveld` | Gewichten met VBW, of voederopname met VFI |
| `dier_id`, `hok` | Diercode EID en hokcode PEN, als tekst |
| `tijd.start`, `tijd.einde` | Geformatteerde bezoekstijdstippen, zonder onbevestigde UTC-offset |
| `waarde`, `eenheid` | VBW in kg of VFI in g per bezoek |
| `hokmetadata` | Gebouw, afdeling en oppervlakte uit jouw tabel hokken |
| `bronrecord` | Volledige oorspronkelijke rij met ongewijzigde veldnamen en waarden |

<div style="background-color:#e8f5e9;color:#1b5e20;border-left:5px solid #2e7d32;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Uitleg</strong><br>Asserva bewaart bijvoorbeeld <code>20230824120000</code>, Schauer gebruikt datum-tijdtekst. De code zet beide om naar een leesbaar formaat zoals <code>2023-08-24T12:00:00</code>. De bron bevat geen tijdzone-offset. Daarom wordt geen <code>Z</code> of UTC-offset toegevoegd. De vermoedelijke Belgische tijdzone blijft als onzekerheid vermeld. Dit verschilt van het huidige oefentijdstip, waarvan de UTC-offset wel bekend is.</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 4.** Zoek het eerste getoonde record terug in DB Browser op je laptop. Vergelijk ROW_ID, EID, PEN en VBW of VFI. Welke bronkolom levert jouw waarde en wat is de eenheid?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 4):**

Voeg hier je antwoord in. Gebruik Browse Data en het filter bij ROW_ID. Noteer jouw record_id en meetwaarde, zodat je vergelijking controleerbaar is.

</div>

## 6. Een meetrecord via de broker volgen

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 04)

Nu schakelen we over naar het echte meettopic (`metingen/...`), zodat echte metingen gescheiden blijven van je eerdere oefenbericht. Je abonneert eerst op dat meettopic, verstuurt de allereerste meting (`eerste`) en leest die meteen terug om te controleren dat alle velden intact aankomen. *(Dit proefbericht wordt nog niet opgeslagen in de verzendadministratie van stap 7, dus je kunt deze cel zo vaak uitvoeren als je wilt.)*

**Communicatie (`Codespaces → DigitalOcean → Codespaces`):** Python stuurt vanuit **Codespaces** één echte meting (die al in het Python-geheugen stond uit stap 5) naar de **Mosquitto-broker op DigitalOcean** en haalt dat bericht via het abonnement weer terug op van de broker.

**Wachtwoord invoeren:** ook bij de cel hieronder verschijnt **bovenaan in het venster** een invoerbalkje. Vul daar jouw **`MQTT-broker eigenaar`**-wachtwoord in en druk op **Enter** (het wachtwoord wordt niet opgeslagen, alleen gebruikt voor de controle op je broker).

</div>

In [ ]:
with MqttSessie(serveradres, ca, getpass("Wachtwoord 'MQTT-broker eigenaar': "), abonnement=topic_metingen) as verbinding:
    verbinding.verstuur(topic_metingen, eerste)
    ontvangen_meting = verbinding.lees()
assert ontvangen_meting["payload"] == eerste
print("Ontvangen topic:", ontvangen_meting["topic"])
print("Bron en record:", ontvangen_meting["payload"]["bron"], ontvangen_meting["payload"]["record_id"])
print("Waarde:", ontvangen_meting["payload"]["waarde"], ontvangen_meting["payload"]["eenheid"])

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Een abonnement kan ook een wildcard-filter bevatten: `+` staat voor één tussenliggend niveau en `#` voor alle volgende niveaus. Het filter `practicum/jouw_deelnemerscode/metingen/+` vangt daardoor beide meettypen onder jouw naam op, zolang je leesrechten op die topics hebt.

**Doe-opdracht:** vervang alleen `abonnement=topic_metingen` in de vorige cel door `abonnement=f"practicum/{deelnemer}/metingen/+"` en voer de cel opnieuw uit (vul bovenaan in het venster opnieuw jouw **`MQTT-broker eigenaar`**-wachtwoord in, want het wordt niet opgeslagen). Laat het topic waarop je publiceert ongewijzigd, want een wildcard gebruik je alleen bij het abonneren. **Communicatie:** er gebeurt dezelfde uitwisseling (`Codespaces → DigitalOcean → Codespaces`), alleen vraagt jouw abonnement nu met een breder filter bij de broker op.

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 5.** Waarom ontvangt het filter met + jouw bericht, terwijl een abonnement op controle/json dat meetbericht niet ontvangt?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 5):**

Voeg hier je antwoord in. Schrijf jouw concrete topic en het passende filter op.

</div>

## 7. Nieuwe records versturen en bewust herpubliceren

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 04)

Wanneer je in een later notebook extra metingen aan jouw database toevoegt, wil je niet steeds opnieuw alle eerdere tienduizenden rijen hoeven versturen. Daarom houdt de code een kleine **verzendadministratie** bij in `gegevens/werk/publicaties_04.db`, waarin per meting wordt vastgelegd dat de broker het bericht heeft bevestigd (jouw werkdatabase zelf blijft onaangeroerd).

Je test hieronder het verschil tussen drie situaties met jouw selectie van twintig records:
1. **Eerste keer versturen:** alle 20 records worden verstuurd (`bevestigd: 20`, `overgeslagen: 0`).
2. **Dezelfde selectie herhalen:** omdat de verzendadministratie ze al kent, worden ze overgeslagen (`bevestigd: 0`, `overgeslagen: 20`).
3. **Bewust herpubliceren (`opnieuw=True`):** de code negeert de verzendadministratie en verstuurt alle 20 records opnieuw (handig als een ontvanger eerder offline was). Een ontvanger in notebook 05 herkent zulke herhalingen aan `bron` + `record_id`, zodat er niets dubbel wordt opgeslagen.

Voer nu eerst Test 1 uit. **Communicatie (`Codespaces → DigitalOcean`, met `PUBACK`-bevestiging terug naar `Codespaces`):** Python stuurt de twintig metingen (die in stap 5 uit jouw werkdatabase in Codespaces zijn gelezen) naar de **Mosquitto-broker op DigitalOcean** en krijgt per bericht een technische bevestiging (`PUBACK`) terug om lokaal bij te houden in `publicaties_04.db`. Er is hier geen subscriber die de meetinhoud zelf terugleest. Vul in het invoerbalkje **bovenaan in het venster** jouw **`MQTT-broker eigenaar`**-wachtwoord in en druk op **Enter** (dit wordt niet opgeslagen, alleen gebruikt om aan te melden bij je broker):

</div>

In [ ]:
administratie = werkmap / "publicaties_04.db"
with MqttSessie(serveradres, ca, getpass("Wachtwoord 'MQTT-broker eigenaar': ")) as verbinding:
    eerste_publicatie = publiceer_selectie(verbinding, topic_metingen, selectie, administratie)
print("Nieuwe records:", eerste_publicatie)

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Test 2:** Bied nu exact dezelfde selectie nog eens aan met dezelfde verzendadministratie. **Communicatie:** Python kijkt eerst lokaal in `gegevens/werk/publicaties_04.db` in **Codespaces**, ziet dat alle twintig records al bevestigd zijn en **stuurt dus geen enkel meetbericht naar de broker op DigitalOcean** (er wordt alleen even aangemeld en meteen weer afgesloten). Vul bovenaan in het venster opnieuw jouw **`MQTT-broker eigenaar`**-wachtwoord in (wordt niet opgeslagen). Je verwacht nul nieuw verstuurde en twintig overgeslagen records (dit blijft ook zo als je eerst de kernel herstart, zolang `publicaties_04.db` bewaard blijft):

</div>

In [ ]:
with MqttSessie(serveradres, ca, getpass("Wachtwoord 'MQTT-broker eigenaar': ")) as verbinding:
    herhaling = publiceer_selectie(verbinding, topic_metingen, selectie, administratie)
assert herhaling == {"geselecteerd": len(selectie), "bevestigd": 0, "overgeslagen": len(selectie)}
print("Zelfde selectie opnieuw aangeboden:", herhaling)

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Test 3:** Herpubliceer nu bewust dezelfde selectie met `opnieuw=True`. **Communicatie (`Codespaces → DigitalOcean`, met `PUBACK`-bevestiging terug naar `Codespaces`):** dit keer negeert Python de lokale verzendadministratie en stuurt het alle twintig metingen opnieuw naar de **Mosquitto-broker op DigitalOcean**, met exact dezelfde `record_id`-waarden, zodat een ontvanger die eerder offline was ze alsnog kan ontvangen. Vul bovenaan in het venster opnieuw jouw **`MQTT-broker eigenaar`**-wachtwoord in (wordt niet opgeslagen). Je verwacht weer twintig bevestigingen:

</div>

In [ ]:
with MqttSessie(serveradres, ca, getpass("Wachtwoord 'MQTT-broker eigenaar': ")) as verbinding:
    herpublicatie = publiceer_selectie(verbinding, topic_metingen, selectie, administratie, opnieuw=True)
assert herpublicatie["bevestigd"] == len(selectie)
print("Bewuste herpublicatie:", herpublicatie)

<div style="background-color:#e8f5e9;color:#1b5e20;border-left:5px solid #2e7d32;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Interpretatie</strong><br>Een abonnement ontvangt berichten alleen terwijl het actief is, omdat we publiceren met <code>retain=False</code>. Een ontvanger die pas later opstart, krijgt dus niet vanzelf eerdere berichten binnen (en zelfs met <code>retain=True</code> bewaart een broker per topic alleen het allerlaatste bericht, geen volledige meetreeks). Daarom starten we in notebook 05 eerst de ontvangers en versturen we pas daarna de volledige database (<code>asserva.db</code> of <code>schauer.db</code>).</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 6.** Noteer de drie aantallen `bevestigd` uit de drie testen hierboven. Waarom kun je twintig brokerbevestigingen krijgen terwijl een ontvanger bij een andere gebruiker toch nul opgeslagen records heeft?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 6):**

Voeg hier je antwoord in. Gebruik jouw eigen tellingen en benoem welke bevestiging je daadwerkelijk hebt gezien.

</div>

<div style="background-color:#e3f2fd;color:#0d47a1;border-left:5px solid #1976d2;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Tip: de volledige database versturen</strong><br>Om later de volledige database in één keer door te geven, kan <code>lees_metingen(..., limiet=None)</code> alle records na elkaar opleveren zonder het werkgeheugen vol te laden, waarna <code>publiceer_selectie(..., opnieuw=True, voortgang=print)</code> ze verstuurt. Dat doen we pas in notebook 05 zodra alle ontvangers klaarstaan. Twintig proefrecords zijn natuurlijk nog niet de volledige database (<code>asserva.db</code> bevat 80.028 meetrijen, <code>schauer.db</code> 146.649).</div>

## 8. Resultaat bewaren en afsluiten

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 04, daarna Source Control)

Het inleverbestand bewaart wat je hebt gecontroleerd: ontvangen oefenwaarde, één teruggelezen bronidentificatie en publicatietellingen. Het bevat geen wachtwoord of serveradres. De verzendadministratie is een werkbestand en blijft buiten Git.

Voer de controlecel uit nadat de ontvangst en de drie publicatiestappen geslaagd zijn. Een groen resultaat controleert hier de uitwisseling met jouw eigen broker. De ontvangst en opslag bij een andere gebruiker controleren we in de volgende oefening.

</div>

In [ ]:
_ontbrekend = [v for v in ("ontvangen_oefening", "ontvangen_aanpassing", "ontvangen_meting", "eerste_publicatie", "herhaling", "herpublicatie") if v not in globals()]
if _ontbrekend:
    raise RuntimeError(
        "Voer eerst alle eerdere testcellen in deze sessie van boven naar beneden uit (ontbreekt in geheugen: "
        + ", ".join(_ontbrekend) + ")."
    )
assert ontvangen_oefening["payload"] == oefenbericht
assert ontvangen_aanpassing["payload"] == aangepast
assert aangepast["waarde"] != oefenbericht["waarde"]
assert ontvangen_meting["payload"] == eerste
assert herhaling["bevestigd"] == 0 and herpublicatie["bevestigd"] == len(selectie)
verslag = {
    "deelnemerscode": deelnemer, "branch": branch, "rol": rol,
    "getest_op_utc": datetime.now(timezone.utc).isoformat(),
    "topic": topic_metingen, "bron": eerste["bron"], "record_id": eerste["record_id"],
    "ontvangen_oefenwaarde": ontvangen_aanpassing["payload"]["waarde"],
    "heen_en_terug": "geslaagd", "nieuwe_records": eerste_publicatie,
    "herhaling": herhaling, "herpublicatie": herpublicatie,
}
inleverpad = hoofdmap / "gegevens/inlevering/04_publicatiecontrole.json"
inleverpad.parent.mkdir(parents=True, exist_ok=True)
inleverpad.write_text(json.dumps(verslag, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Bewaard:", inleverpad.relative_to(hoofdmap))

<div style="background-color:#fff8e1;color:#6d4c00;border-left:5px solid #f9a825;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Let op: stop je Codespace</strong><br>Een geopende Codespace blijft rekentijd gebruiken. Sla je werk op, commit en push vóór je de Codespace stopt. De Droplet heeft afzonderlijke kosten. Sluit of verwijder die alleen volgens de afspraak met de docent.</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

1. Vul je antwoorden aan met jouw uitvoer. Kies **File → Save All**.
2. Open **Source Control**. Selecteer notebook 04 en `gegevens/inlevering/04_publicatiecontrole.json` met het **+**-teken. Controleer dat geen wachtwoord in een cel of uitvoer staat.
3. Commit met bijvoorbeeld **MQTT-publicatie en herpublicatie gecontroleerd** en kies **Push**. Controleer op GitHub jouw branch en beide bestanden.
4. Bewaar bij het verwijderen van een Codespace eerst je werkdatabase en `publicaties_04.db` apart op je laptop. De verzendadministratie is na deze cellen gesloten. Bij verlies ervan kan de code dezelfde records opnieuw versturen. De ontvanger moet herhalingen kunnen herkennen.
5. Je kunt de broker na de oefening handmatig stoppen met onderstaande opdracht in de SSH-terminal (spreek dit even af als een andere gebruiker nog met jouw broker verbindt). Start hem bij de volgende oefening weer handmatig.
6. Kies via [github.com/codespaces](https://github.com/codespaces) **Stop codespace**. Hervat later dezelfde Codespace.

</div>

```bash
sudo systemctl stop mosquitto
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

- [ ] Ik kan publisher, subscriber, broker, topic en payload onderscheiden.
- [ ] Ik heb een JSON-waarde aangepast en de ontvangen wijziging gezien.
- [ ] Ik heb een echte bronrij teruggevonden en via MQTT teruggelezen.
- [ ] Ik heb een topicfilter met `+` geprobeerd en verklaard.
- [ ] Ik kan nieuwe publicatie, herhaling en bewuste herpublicatie verklaren.
- [ ] Mijn antwoorden en controleverslag staan op mijn eigen GitHub-branch.
- [ ] Mijn werkbestanden zijn bewaard en mijn Codespace is gestopt.

</div>

## Hulp bij veelvoorkomende problemen

| Probleem | Controle en herstel |
|---|---|
| Werkbestand ontbreekt | Hervat notebook 02 voor de database of notebook 03 voor server.json en ca.crt. Gebruik geen lege vervangbestanden. |
| Verkeerde rol of ontbrekende meetkolom | Controleer je rol uit notebook 01 en de gekozen werkdatabase. VBW hoort bij gewichten, VFI bij voederopname. |
| Hokmetadata ontbreken | Controleer of je werkdatabase de tabel hokken uit notebook 02 bevat, met de juiste hokcodes. |
| Connection refused of time-out | Controleer serveradres, poort 8883 en of Mosquitto handmatig is gestart. Controleer de firewall volgens notebook 03. |
| Certificaatfout | Controleer het serveradres en het via SSH gekopieerde CA-bestand. Bij een verlopen certificaat vraag je hulp. Schakel certificaatcontrole niet uit. |
| Aanmelding geweigerd | Gebruik het wachtwoord **`MQTT-broker eigenaar`**, niet het **`SSH beheer-wachtwoord`** of **`API-Droplet wachtwoord`**. |
| Publicatie geweigerd | Controleer de deelnemerscode in het topic en in het rechtenbestand uit notebook 03. Account `lezer` (**`MQTT-broker lezer`**) heeft geen schrijfrechten. |
| Geen ontvangst | Controleer of abonnement en topic overeenkomen. De ontvangende cel moet actief zijn tijdens het publiceren. |
| Alles wordt overgeslagen | De administratie kent deze records al. Gebruik de herpublicatiecel om ze bewust nogmaals te versturen. |
| Eerder record of metadata gewijzigd | De oefening deelt toevoegingen. Bewaar je bestanden en bespreek de wijziging. Verwijder de administratie niet om de melding te omzeilen. |
| Onderbroken publicatie | Herstel de verbinding en bied de selectie opnieuw aan. Enkele berichten kunnen herhaald worden. Voor ontbrekende historie gebruik je bewuste herpublicatie. |
| NameError na kernelherstart | Voer de voorgaande cellen opnieuw uit. Variabelen bestaan alleen tijdens de huidige kernelsessie. |

## Technische bronnen

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

- [Eclipse Paho: Python-client, callbacks en publicatiebevestiging](https://eclipse.dev/paho/files/paho.mqtt.python/html/client.html)
- [Eclipse Mosquitto: topics, abonnementen, QoS en retained berichten](https://mosquitto.org/man/mqtt-7.html)
- [Python: JSON-omzetting](https://docs.python.org/3/library/json.html)

De verbindingscode gebruikt de Paho-callbackinterface versie 2 en MQTT 5. De technische documentatie is gecontroleerd op 18 november 2026.

</div>